In [ ]:
from __future__ import annotations
import json
import re
from pathlib import Path
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm
SNAPSHOT_ROOT = Path('/mnt/primary/Finnhub Pipeline/finnhub_snapshots')
OUTPUT_ROOT = Path('/mnt/primary/Finnhub Pipeline/finnhub_snapshots_filtered')
COMPANY_CSV = Path('/mnt/primary/sp500.csv')
START_DATE = pd.Timestamp('2026-07-15')
END_DATE = pd.Timestamp('2026-08-28')
MATCH_TICKER = True
ALLOW_SINGLE_CHARACTER_TICKERS = False
OVERWRITE_ORIGINALS = False
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Input root :', SNAPSHOT_ROOT)
print('Output root:', OUTPUT_ROOT)
print('Start date :', START_DATE.date())

In [ ]:
companies = pd.read_csv(COMPANY_CSV)
required_columns = {'Symbol', 'Security'}
missing = required_columns - set(companies.columns)
if missing:
    raise ValueError(f'Company CSV is missing columns: {sorted(missing)}')
companies['Symbol'] = companies['Symbol'].astype(str).str.strip().str.upper()
companies['Security'] = companies['Security'].astype(str).str.strip()
ticker_to_company = dict(zip(companies['Symbol'], companies['Security']))
print('Company mappings loaded:', len(ticker_to_company))
display(companies[['Symbol', 'Security']].head())

In [ ]:
CORPORATE_SUFFIX_PATTERN = re.compile('\n    \\s+\n    (?:\n        Incorporated |\n        Corporation |\n        Corp\\.? |\n        Company |\n        Co\\.? |\n        Holdings? |\n        Holding |\n        Group |\n        Inc\\.? |\n        Ltd\\.? |\n        Limited |\n        PLC |\n        plc |\n        N\\.?V\\.? |\n        S\\.?A\\.?\n    )\n    \\s*$\n    ', re.IGNORECASE | re.VERBOSE)

def normalise_space(text: str) -> str:
    return re.sub('\\s+', ' ', str(text)).strip()

def remove_share_class(company_name: str) -> str:
    return re.sub('\\s*\\([^)]*(?:Class|Share)[^)]*\\)\\s*$', '', company_name, flags=re.IGNORECASE).strip()

def remove_corporate_suffix(company_name: str) -> str:
    previous = None
    current = company_name.strip()
    while previous != current:
        previous = current
        current = CORPORATE_SUFFIX_PATTERN.sub('', current).strip()
    return current

def build_company_aliases(company_name: str, ticker: str, profile_name: str | None=None) -> list[str]:
    aliases = set()
    names = [company_name, profile_name]
    for name in names:
        if not name:
            continue
        name = normalise_space(name)
        aliases.add(name)
        without_class = remove_share_class(name)
        if without_class:
            aliases.add(without_class)
        short_name = remove_corporate_suffix(without_class)
        if short_name:
            aliases.add(short_name)
    ticker = str(ticker).strip().upper()
    if MATCH_TICKER and ticker:
        if len(ticker) > 1 or ALLOW_SINGLE_CHARACTER_TICKERS:
            aliases.add(ticker)
        aliases.add(f'${ticker}')
    aliases = {alias for alias in aliases if len(alias.strip()) >= 2}
    return sorted(aliases, key=len, reverse=True)

In [ ]:
def headline_contains_alias(headline: str, alias: str) -> bool:
    if not headline:
        return False
    headline = str(headline)
    pattern = '(?<![A-Za-z0-9])' + re.escape(alias) + '(?![A-Za-z0-9])'
    return bool(re.search(pattern, headline, flags=re.IGNORECASE))

def headline_directly_mentions_company(headline: str, aliases: list[str]) -> tuple[bool, str | None]:
    for alias in aliases:
        if headline_contains_alias(headline, alias):
            return (True, alias)
    return (False, None)

In [ ]:
def get_snapshot_identity(snapshot: dict, snapshot_path: Path) -> tuple[str | None, str | None, str | None]:
    ticker = snapshot.get('ticker')
    company_name = snapshot.get('company_name')
    profile = snapshot.get('data', {}).get('company_profile', {}) or {}
    profile_ticker = profile.get('ticker')
    profile_name = profile.get('name')
    if not ticker:
        ticker = profile_ticker
    if not company_name:
        company_name = profile_name
    if not ticker:
        filename = snapshot_path.stem
        match = re.match('^([A-Za-z0-9.\\-]+)_snapshot$', filename)
        if match:
            ticker = match.group(1).upper()
    if ticker:
        ticker = str(ticker).strip().upper()
        if not company_name:
            company_name = ticker_to_company.get(ticker)
    return (company_name, ticker, profile_name)

In [ ]:
def filter_snapshot_news(snapshot_path: Path) -> dict:
    with open(snapshot_path, 'r', encoding='utf-8') as file:
        snapshot = json.load(file)
    company_name, ticker, profile_name = get_snapshot_identity(snapshot, snapshot_path)
    if not ticker:
        raise ValueError(f'Could not determine ticker for {snapshot_path}')
    if not company_name:
        raise ValueError(f'Could not determine company name for {ticker}: {snapshot_path}')
    aliases = build_company_aliases(company_name=company_name, ticker=ticker, profile_name=profile_name)
    data = snapshot.setdefault('data', {})
    original_news = data.get('company_news', []) or []
    filtered_news = []
    for article in original_news:
        headline = article.get('headline', '') or ''
        matched, matched_alias = headline_directly_mentions_company(headline, aliases)
        if not matched:
            continue
        filtered_article = article.copy()
        filtered_article['direct_company_match'] = True
        filtered_article['matched_company_alias'] = matched_alias
        filtered_news.append(filtered_article)
    data['company_news'] = filtered_news
    snapshot['news_filter'] = {'method': 'headline_company_name_or_ticker', 'company_name': company_name, 'ticker': ticker, 'aliases_used': aliases, 'original_news_count': len(original_news), 'filtered_news_count': len(filtered_news), 'removed_news_count': len(original_news) - len(filtered_news)}
    return snapshot

In [ ]:
def is_date_folder(path: Path) -> bool:
    if not path.is_dir():
        return False
    try:
        pd.Timestamp(path.name)
        return True
    except Exception:
        return False
date_folders = []
for folder in SNAPSHOT_ROOT.iterdir():
    if not is_date_folder(folder):
        continue
    folder_date = pd.Timestamp(folder.name)
    if folder_date < START_DATE:
        continue
    if END_DATE is not None and folder_date > END_DATE:
        continue
    date_folders.append((folder_date, folder))
date_folders = sorted(date_folders, key=lambda item: item[0])
print('Date folders found:', len(date_folders))
if date_folders:
    print('First:', date_folders[0][0].date())
    print('Last:', date_folders[-1][0].date())

In [ ]:
processing_log = []
for folder_date, source_folder in tqdm(date_folders, desc='Processing dates'):
    if OVERWRITE_ORIGINALS:
        destination_folder = source_folder
    else:
        destination_folder = OUTPUT_ROOT / source_folder.name
        destination_folder.mkdir(parents=True, exist_ok=True)
    snapshot_files = sorted(source_folder.glob('*.json'))
    for snapshot_path in snapshot_files:
        try:
            filtered_snapshot = filter_snapshot_news(snapshot_path)
            filter_info = filtered_snapshot['news_filter']
            if OVERWRITE_ORIGINALS:
                output_path = snapshot_path
            else:
                output_path = destination_folder / snapshot_path.name
            with open(output_path, 'w', encoding='utf-8') as file:
                json.dump(filtered_snapshot, file, indent=2, ensure_ascii=False)
            processing_log.append({'Date': folder_date.date(), 'Ticker': filter_info['ticker'], 'Company': filter_info['company_name'], 'InputFile': str(snapshot_path), 'OutputFile': str(output_path), 'OriginalNews': filter_info['original_news_count'], 'KeptNews': filter_info['filtered_news_count'], 'RemovedNews': filter_info['removed_news_count'], 'Status': 'success'})
        except Exception as error:
            processing_log.append({'Date': folder_date.date(), 'Ticker': None, 'Company': None, 'InputFile': str(snapshot_path), 'OutputFile': None, 'OriginalNews': None, 'KeptNews': None, 'RemovedNews': None, 'Status': 'error', 'Error': f'{type(error).__name__}: {error}'})

In [ ]:
log_df = pd.DataFrame(processing_log)
display(log_df.head(20))
print('=' * 70)
print('FILTERING COMPLETE')
print('=' * 70)
print('Snapshots processed:', len(log_df))
print('Successful:', (log_df['Status'] == 'success').sum())
print('Errors:', (log_df['Status'] == 'error').sum())
successful = log_df[log_df['Status'] == 'success']
if not successful.empty:
    print()
    print('Original news articles:', int(successful['OriginalNews'].sum()))
    print('News articles kept:', int(successful['KeptNews'].sum()))
    print('News articles removed:', int(successful['RemovedNews'].sum()))
    keep_rate = successful['KeptNews'].sum() / successful['OriginalNews'].sum() * 100
    print(f'Overall keep rate: {keep_rate:.2f}%')
log_path = OUTPUT_ROOT / 'news_filtering_log.csv'
log_df.to_csv(log_path, index=False)
print()
print('Log saved:', log_path)